# Tomahawk Fusion — CuCrZr RF-Cassette Coolant Topology Screen

**Purpose:** preliminary Kaggle screening of parallel, ribbed/hypervapotron-inspired,
circular-control, and microchannel-stress-test cooling geometries for the RF cassette.

The notebook runs a small smoke screen first and then automatically runs the full
parameter sweep. It writes CSV summaries and PNG plots to `/kaggle/working` when run
on Kaggle (or to the current directory elsewhere).

**Important scope boundary:** this is a reduced-order engineering screen. It is not
a validated CFD, OpenMC/MCNP neutronics, two-phase boiling, RF electromagnetic, or
structural qualification model. Results are preliminary and idealized.

In [1]:
# SECTION 1 — Kaggle-safe imports and run configuration
import itertools
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260824
np.random.seed(SEED)

if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working")
else:
    OUT_DIR = Path.cwd() / "tomahawk_channel_screen_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Upload-and-run behavior: smoke is always executed before full.
RUN_MODE = "smoke_then_full"
print("Output directory:", OUT_DIR)
print("Run mode:", RUN_MODE, "| Seed:", SEED)
print("Dependencies: numpy, pandas, matplotlib, Python standard library only")

Output directory: /kaggle/working
Run mode: smoke_then_full | Seed: 20260824
Dependencies: numpy, pandas, matplotlib, Python standard library only


## Governing screening relations

Heat load:

`Q_total = Q_nuclear + Q_RF_wall`

`Q_RF_wall = P_RF × wall_loss_fraction`

`m_dot = Q_total / (cp × ΔT_coolant)`

Channel hydraulics:

`D_h = 4 A_c / P_w`

`Re = rho v D_h / mu`

`Δp = f (L/D_h) (rho v²/2) + ΣK (rho v²/2)`

Heat-transfer screening:

`Nu = 0.023 Re^0.8 Pr^0.4` for turbulent smooth-channel screening,
`h = Nu k / D_h`, and `T_wall = T_in + ΔT_coolant + q''(1/h + t_Cu/k_Cu)`.

Rib multipliers are phenomenological proxies only. They must be replaced by
geometry-specific CFD correlations and, if applicable, a validated two-phase model.

In [2]:
# SECTION 2 — Revision 2.1 inputs and explicit assumptions
INPUTS = {
    # Plant/module geometry
    "n_modules_active": 10,
    "n_cassettes_per_module": 4,
    "cassette_area_m2": 2.5,       # 10 m^2/module divided into four cassettes
    "cucrz_thickness_m": 0.010,    # swept later in the full run
    "cucrz_k_W_mK": 300.0,         # screening value; irradiation-dependent in reality

    # Neutron/RF heat assumptions
    "neutron_wall_load_MW_m2": 4.0,
    "cucrz_qvol_screen_MW_m3": [50.0, 100.0, 200.0],
    "rf_frequency_MHz": 107.4,
    "rf_power_absorbed_MW": 15.0,
    "rf_wall_loss_fraction": [0.02, 0.05, 0.10],

    # Pressurized subcooled-water screen
    "rho_kg_m3": 950.0,
    "mu_Pa_s": 2.30e-4,
    "k_W_mK": 0.68,
    "cp_J_kgK": 4180.0,
    "pressure_MPa": [3.0, 4.0, 5.0],
    "inlet_temp_C": [100.0, 125.0, 150.0],
    "deltaT_C": [20.0, 30.0, 40.0],

    # Parametric channel geometry; equivalent square area is used for the screen
    "channel_count": [16, 32, 64],
    "Dh_mm": [8.0, 12.0, 16.0],
    "channel_length_m": 1.5,
    "minor_loss_K": 4.0,

    # Stop conditions for preliminary screening
    "max_pressure_drop_MPa": 1.0,
    "max_velocity_m_s": 5.0,
    "max_CuCrZr_temperature_C": 300.0,
    "boiling_margin_C": 10.0,
}

# Heat-load branches prevent accidental double counting. The default wall-load
# branch uses 4 MW/m^2; the volumetric branch is a separate sensitivity case.
HEAT_LOAD_MODES = ["wall_load", "volumetric_CuCrZr"]

# Topology multipliers are intentionally exposed and easy to replace with CFD data.
TOPOLOGIES = {
    "parallel_smooth": {
        "nu_multiplier": 1.00,
        "dp_multiplier": 1.00,
        "description": "short symmetric parallel channels",
    },
    "parallel_ribbed": {
        "nu_multiplier": 1.25,
        "dp_multiplier": 1.60,
        "description": "parallel channels with localized low-profile cross-ribs",
    },
    "circular_control": {
        "nu_multiplier": 0.95,
        "dp_multiplier": 1.00,
        "description": "smooth circular-routing control case",
    },
    "microchannel_stress_test": {
        "nu_multiplier": 1.35,
        "dp_multiplier": 3.00,
        "description": "small-channel stress case; not the recommended first CAD",
    },
}

print(json.dumps({
    "RF_frequency_MHz": INPUTS["rf_frequency_MHz"],
    "RF_power_absorbed_MW_per_module": INPUTS["rf_power_absorbed_MW"],
    "neutron_wall_load_MW_m2": INPUTS["neutron_wall_load_MW_m2"],
    "pressure_range_MPa": INPUTS["pressure_MPa"],
    "inlet_temperature_C": INPUTS["inlet_temp_C"],
    "topologies": list(TOPOLOGIES),
}, indent=2))

{
  "RF_frequency_MHz": 107.4,
  "RF_power_absorbed_MW_per_module": 15.0,
  "neutron_wall_load_MW_m2": 4.0,
  "pressure_range_MPa": [
    3.0,
    4.0,
    5.0
  ],
  "inlet_temperature_C": [
    100.0,
    125.0,
    150.0
  ],
  "topologies": [
    "parallel_smooth",
    "parallel_ribbed",
    "circular_control",
    "microchannel_stress_test"
  ]
}


In [3]:
# SECTION 3 — Reduced-order property and topology functions
def saturation_temperature_C(pressure_MPa):
    """Approximate water saturation temperature for a screening margin only."""
    p = float(pressure_MPa)
    # Approximate values near the specified 3–5 MPa range.
    return float(np.interp(p, [3.0, 4.0, 5.0], [233.9, 250.4, 263.9]))


def darcy_friction_factor(Re):
    Re = max(float(Re), 1.0)
    if Re < 2300.0:
        return 64.0 / Re
    return 0.3164 / (Re ** 0.25)


def evaluate_case(
    topology,
    heat_load_mode,
    qvol_MW_m3,
    rf_loss_fraction,
    pressure_MPa,
    inlet_temp_C,
    deltaT_C,
    channel_count,
    Dh_mm,
    cu_thickness_m,
):
    topology_name = topology
    topo = TOPOLOGIES[topology_name]
    rho = INPUTS["rho_kg_m3"]
    mu = INPUTS["mu_Pa_s"]
    k_water = INPUTS["k_W_mK"]
    cp = INPUTS["cp_J_kgK"]
    n_cass = INPUTS["n_cassettes_per_module"]
    cassette_area = INPUTS["cassette_area_m2"]
    total_area = n_cass * cassette_area

    q_rf_wall_W = INPUTS["rf_power_absorbed_MW"] * 1e6 * rf_loss_fraction
    if heat_load_mode == "wall_load":
        q_nuclear_areal_W_m2 = INPUTS["neutron_wall_load_MW_m2"] * 1e6
        q_nuclear_W = q_nuclear_areal_W_m2 * total_area
    else:
        q_nuclear_areal_W_m2 = qvol_MW_m3 * 1e6 * cu_thickness_m
        q_nuclear_W = q_nuclear_areal_W_m2 * total_area

    q_total_W = q_nuclear_W + q_rf_wall_W
    m_dot_module_kg_s = q_total_W / (cp * deltaT_C)
    q_cassette_W = q_total_W / n_cass
    m_dot_cassette_kg_s = m_dot_module_kg_s / n_cass

    Dh = Dh_mm / 1000.0
    # Square-equivalent channel area; replace with CAD cross-sectional area later.
    A_channel_m2 = Dh ** 2
    A_flow_total_m2 = channel_count * A_channel_m2
    v = m_dot_cassette_kg_s / (rho * A_flow_total_m2)
    Re = rho * v * Dh / mu
    Pr = cp * mu / k_water

    if Re < 2300.0:
        Nu_base = 4.36
    else:
        Nu_base = 0.023 * (Re ** 0.8) * (Pr ** 0.4)
    Nu = Nu_base * topo["nu_multiplier"]
    h = Nu * k_water / Dh

    f = darcy_friction_factor(Re) * topo["dp_multiplier"]
    dynamic_pressure = 0.5 * rho * v * v
    dp_friction = f * (INPUTS["channel_length_m"] / Dh) * dynamic_pressure
    dp_minor = INPUTS["minor_loss_K"] * topo["dp_multiplier"] * dynamic_pressure
    dp_total_Pa = dp_friction + dp_minor

    q_face_W_m2 = q_cassette_W / cassette_area
    dt_convective_C = q_face_W_m2 / max(h, 1e-12)
    dt_conduction_C = q_face_W_m2 * cu_thickness_m / INPUTS["cucrz_k_W_mK"]
    wall_temp_C = inlet_temp_C + deltaT_C + dt_convective_C + dt_conduction_C
    Tsat_C = saturation_temperature_C(pressure_MPa)

    pass_thermal = wall_temp_C <= INPUTS["max_CuCrZr_temperature_C"]
    pass_dp = dp_total_Pa <= INPUTS["max_pressure_drop_MPa"] * 1e6
    pass_velocity = v <= INPUTS["max_velocity_m_s"]
    pass_single_phase = wall_temp_C <= Tsat_C - INPUTS["boiling_margin_C"]
    passes_all = bool(pass_thermal and pass_dp and pass_velocity and pass_single_phase)

    return {
        "topology": topology_name,
        "topology_description": topo["description"],
        "heat_load_mode": heat_load_mode,
        "qvol_MW_m3": qvol_MW_m3,
        "rf_loss_fraction": rf_loss_fraction,
        "pressure_MPa": pressure_MPa,
        "inlet_temp_C": inlet_temp_C,
        "deltaT_C": deltaT_C,
        "channel_count": channel_count,
        "Dh_mm": Dh_mm,
        "cu_thickness_mm": cu_thickness_m * 1000.0,
        "q_nuclear_areal_MW_m2": q_nuclear_areal_W_m2 / 1e6,
        "q_RF_wall_MW": q_rf_wall_W / 1e6,
        "q_total_module_MW": q_total_W / 1e6,
        "m_dot_module_kg_s": m_dot_module_kg_s,
        "m_dot_cassette_kg_s": m_dot_cassette_kg_s,
        "flow_area_cassette_m2": A_flow_total_m2,
        "velocity_m_s": v,
        "Re": Re,
        "Pr": Pr,
        "Nu": Nu,
        "h_W_m2K": h,
        "pressure_drop_kPa": dp_total_Pa / 1000.0,
        "wall_temp_C": wall_temp_C,
        "saturation_temp_C": Tsat_C,
        "single_phase_margin_C": Tsat_C - wall_temp_C,
        "thermal_margin_C": INPUTS["max_CuCrZr_temperature_C"] - wall_temp_C,
        "pass_thermal": pass_thermal,
        "pass_pressure_drop": pass_dp,
        "pass_velocity": pass_velocity,
        "pass_single_phase": pass_single_phase,
        "passes_all_gates": passes_all,
    }


def build_parameter_grid(mode="smoke"):
    if mode == "smoke":
        return {
            "topology": list(TOPOLOGIES),
            "heat_load_mode": ["wall_load", "volumetric_CuCrZr"],
            "qvol_MW_m3": [100.0],
            "rf_loss_fraction": [0.05],
            "pressure_MPa": [4.0],
            "inlet_temp_C": [125.0],
            "deltaT_C": [30.0],
            "channel_count": [32],
            "Dh_mm": [12.0],
            "cu_thickness_m": [0.010],
        }
    return {
        "topology": list(TOPOLOGIES),
        "heat_load_mode": HEAT_LOAD_MODES,
        "qvol_MW_m3": INPUTS["cucrz_qvol_screen_MW_m3"],
        "rf_loss_fraction": INPUTS["rf_wall_loss_fraction"],
        "pressure_MPa": INPUTS["pressure_MPa"],
        "inlet_temp_C": INPUTS["inlet_temp_C"],
        "deltaT_C": INPUTS["deltaT_C"],
        "channel_count": INPUTS["channel_count"],
        "Dh_mm": INPUTS["Dh_mm"],
        "cu_thickness_m": [0.006, 0.010, 0.015],
    }

In [4]:
# SECTION 4 — Automatic smoke run followed by full run
def run_sweep(mode):
    grid = build_parameter_grid(mode)
    keys = list(grid)
    values = [grid[k] for k in keys]
    rows = []
    start = time.time()
    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))
        rows.append(evaluate_case(**params))
    result = pd.DataFrame(rows)
    result["run_mode"] = mode
    result["elapsed_s"] = time.time() - start
    return result


smoke_df = run_sweep("smoke")
print("Smoke rows:", len(smoke_df))
print("Smoke pass count:", int(smoke_df["passes_all_gates"].sum()))
display(smoke_df[[
    "topology", "heat_load_mode", "m_dot_module_kg_s", "velocity_m_s",
    "pressure_drop_kPa", "wall_temp_C", "thermal_margin_C",
    "single_phase_margin_C", "passes_all_gates"
]].round(3))

full_df = run_sweep("full")
print("Full rows:", len(full_df))
print("Full runtime (s):", round(float(full_df["elapsed_s"].iloc[0]), 2))
print("Full pass count:", int(full_df["passes_all_gates"].sum()))

Smoke rows: 8
Smoke pass count: 4


,topology,heat_load_mode,m_dot_module_kg_s,velocity_m_s,pressure_drop_kPa,wall_temp_C,thermal_margin_C,single_phase_margin_C,passes_all_gates
0,parallel_smooth,wall_load,324.960,18.558,863.288,336.959,-36.959,-86.559,False
1,parallel_smooth,volumetric_CuCrZr,85.726,4.896,65.826,226.168,73.832,24.232,True
2,parallel_ribbed,wall_load,324.960,18.558,1381.260,327.734,-27.734,-77.334,False
3,parallel_ribbed,volumetric_CuCrZr,85.726,4.896,105.322,219.101,80.899,31.299,True
4,circular_control,wall_load,324.960,18.558,863.288,339.387,-39.387,-88.987,False
5,circular_control,volumetric_CuCrZr,85.726,4.896,65.826,228.028,71.972,22.372,True
6,microchannel_stress_test,wall_load,324.960,18.558,2589.863,325.001,-25.001,-74.601,False
7,microchannel_stress_test,volumetric_CuCrZr,85.726,4.896,197.479,217.007,82.993,33.393,True


Full rows: 52488
Full runtime (s): 1.04
Full pass count: 7018


## Interpretation gate

A row passes only when all four reduced-order gates pass:

1. CuCrZr wall temperature ≤ 300 °C.
2. Cassette pressure drop ≤ 1 MPa.
3. Channel velocity ≤ 5 m/s.
4. Estimated wall temperature remains at least 10 °C below the approximate water
   saturation temperature at the selected pressure.

A passing row is **not** proof of reactor feasibility. It only identifies a candidate
geometry for higher-fidelity CFD, neutronics, RF-loss mapping, and materials testing.

In [5]:
# SECTION 5 — Results, feasibility ranking, and flow-area sanity check
numeric_cols = [
    "q_nuclear_areal_MW_m2", "q_total_module_MW", "m_dot_module_kg_s",
    "m_dot_cassette_kg_s", "flow_area_cassette_m2", "velocity_m_s", "Re",
    "h_W_m2K", "pressure_drop_kPa", "wall_temp_C", "thermal_margin_C",
    "single_phase_margin_C"
]
full_df[numeric_cols] = full_df[numeric_cols].astype(float)

feasible = full_df[full_df["passes_all_gates"]].copy()
if len(feasible):
    feasible["ranking_score"] = (
        feasible["thermal_margin_C"]
        + 0.15 * feasible["single_phase_margin_C"]
        - 0.05 * feasible["pressure_drop_kPa"]
        - 2.0 * np.maximum(feasible["velocity_m_s"] - 3.0, 0.0)
    )
    ranked = feasible.sort_values("ranking_score", ascending=False)
    print("Top feasible candidates:")
    display(ranked.head(20)[[
        "topology", "heat_load_mode", "qvol_MW_m3", "rf_loss_fraction",
        "pressure_MPa", "inlet_temp_C", "deltaT_C", "channel_count", "Dh_mm",
        "m_dot_module_kg_s", "velocity_m_s", "pressure_drop_kPa", "wall_temp_C",
        "thermal_margin_C", "single_phase_margin_C", "ranking_score"
    ]].round(3))
else:
    ranked = feasible
    print("No rows passed all reduced-order gates. Inspect the least-bad rows below.")
    display(full_df.sort_values(
        ["wall_temp_C", "pressure_drop_kPa"]
    ).head(20)[[
        "topology", "heat_load_mode", "q_total_module_MW", "m_dot_module_kg_s",
        "velocity_m_s", "pressure_drop_kPa", "wall_temp_C", "thermal_margin_C",
        "single_phase_margin_C"
    ]].round(3))

# This check directly exposes whether the current 63.96 kg/s/module screen is
# compatible with small channels. It is intentionally printed before CAD sizing.
q_ref_MW = 8.02
mdot_ref = q_ref_MW * 1e6 / (INPUTS["cp_J_kgK"] * 30.0)
mdot_ref_cassette = mdot_ref / INPUTS["n_cassettes_per_module"]
area_at_2ms = mdot_ref_cassette / (INPUTS["rho_kg_m3"] * 2.0)
area_at_5ms = mdot_ref_cassette / (INPUTS["rho_kg_m3"] * 5.0)
area_3mm = math.pi * (0.003 / 2.0) ** 2
n_3mm_at_2ms = mdot_ref_cassette / (INPUTS["rho_kg_m3"] * 2.0 * area_3mm)
n_3mm_at_5ms = mdot_ref_cassette / (INPUTS["rho_kg_m3"] * 5.0 * area_3mm)
print("\nFlow-area sanity check using the earlier 8.02 MW/module, ΔT=30 °C screen:")
print("  Reference module flow [kg/s]:", round(mdot_ref, 3))
print("  Reference cassette flow [kg/s]:", round(mdot_ref_cassette, 3))
print("  Required flow area/cassette at 2 m/s [m^2]:", round(area_at_2ms, 6))
print("  Required flow area/cassette at 5 m/s [m^2]:", round(area_at_5ms, 6))
print("  Equivalent 3 mm circular channels required at 2 m/s:", round(n_3mm_at_2ms))
print("  Equivalent 3 mm circular channels required at 5 m/s:", round(n_3mm_at_5ms))
print("  Interpretation: current screen flow is not compatible with a small number of 2–4 mm channels.")

Top feasible candidates:


,topology,heat_load_mode,qvol_MW_m3,rf_loss_fraction,pressure_MPa,inlet_temp_C,deltaT_C,channel_count,Dh_mm,m_dot_module_kg_s,velocity_m_s,pressure_drop_kPa,wall_temp_C,thermal_margin_C,single_phase_margin_C,ranking_score
20187,parallel_ribbed,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,64,8.0,39.474,2.536,36.596,140.143,159.857,123.757,176.591
46431,microchannel_stress_test,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,64,8.0,39.474,2.536,68.618,139.139,160.861,124.761,176.144
20172,parallel_ribbed,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,16,12.0,39.474,4.509,89.898,135.869,164.131,128.031,175.824
46419,microchannel_stress_test,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,16,16.0,39.474,2.536,50.097,141.004,158.996,122.896,174.926
7050,parallel_smooth,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,16,12.0,39.474,4.509,56.186,138.186,161.814,125.714,174.845
20175,parallel_ribbed,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,16,16.0,39.474,2.536,26.719,142.156,157.844,121.744,174.769
20916,parallel_ribbed,volumetric_CuCrZr,50.0,0.05,5.0,100.0,20.0,64,8.0,44.856,2.882,46.565,141.393,158.607,122.507,174.655
46425,microchannel_stress_test,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,32,12.0,39.474,2.254,44.632,141.542,158.458,122.358,174.580
19944,parallel_ribbed,volumetric_CuCrZr,50.0,0.02,4.0,100.0,20.0,64,8.0,39.474,2.536,36.596,140.143,159.857,110.257,174.566
20181,parallel_ribbed,volumetric_CuCrZr,50.0,0.02,5.0,100.0,20.0,32,12.0,39.474,2.254,23.804,142.738,157.262,121.162,174.247



Flow-area sanity check using the earlier 8.02 MW/module, ΔT=30 °C screen:
  Reference module flow [kg/s]: 63.955
  Reference cassette flow [kg/s]: 15.989
  Required flow area/cassette at 2 m/s [m^2]: 0.008415
  Required flow area/cassette at 5 m/s [m^2]: 0.003366
  Equivalent 3 mm circular channels required at 2 m/s: 1191
  Equivalent 3 mm circular channels required at 5 m/s: 476
  Interpretation: current screen flow is not compatible with a small number of 2–4 mm channels.


In [6]:
# SECTION 6 — Plots and machine-readable outputs
summary = full_df.groupby("topology", as_index=False).agg(
    cases=("topology", "size"),
    passing_cases=("passes_all_gates", "sum"),
    minimum_wall_temp_C=("wall_temp_C", "min"),
    maximum_wall_temp_C=("wall_temp_C", "max"),
    minimum_pressure_drop_kPa=("pressure_drop_kPa", "min"),
    maximum_pressure_drop_kPa=("pressure_drop_kPa", "max"),
    maximum_velocity_m_s=("velocity_m_s", "max"),
)
summary["pass_fraction"] = summary["passing_cases"] / summary["cases"]

full_path = OUT_DIR / "tomahawk_channel_topology_full_results.csv"
summary_path = OUT_DIR / "tomahawk_channel_topology_summary.csv"
manifest_path = OUT_DIR / "tomahawk_channel_topology_run_manifest.json"
plot_path = OUT_DIR / "tomahawk_channel_topology_wall_vs_dp.png"

full_df.to_csv(full_path, index=False)
summary.to_csv(summary_path, index=False)
manifest = {
    "seed": SEED,
    "run_mode": RUN_MODE,
    "notebook_scope": "reduced-order coolant topology screen",
    "inputs": INPUTS,
    "topologies": TOPOLOGIES,
    "outputs": [str(full_path), str(summary_path), str(plot_path)],
    "warning": "Preliminary screening only; not validated CFD/neutronics/materials qualification.",
}
manifest_path.write_text(json.dumps(manifest, indent=2))

plt.figure(figsize=(10, 6))
for name, group in full_df.groupby("topology"):
    plt.scatter(
        group["pressure_drop_kPa"], group["wall_temp_C"],
        s=10, alpha=0.35, label=name
    )
plt.axhline(INPUTS["max_CuCrZr_temperature_C"], color="red", linestyle="--", label="300 °C limit")
plt.axvline(INPUTS["max_pressure_drop_MPa"] * 1000.0, color="black", linestyle=":", label="1 MPa limit")
plt.xlabel("Cassette pressure drop [kPa]")
plt.ylabel("Estimated CuCrZr wall temperature [°C]")
plt.title("Tomahawk RF-cassette topology screen")
plt.grid(alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(plot_path, dpi=160)
plt.close()

print("\nSaved:")
print(" ", full_path)
print(" ", summary_path)
print(" ", manifest_path)
print(" ", plot_path)
display(summary.round(3))


Saved:
  /kaggle/working/tomahawk_channel_topology_full_results.csv
  /kaggle/working/tomahawk_channel_topology_summary.csv
  /kaggle/working/tomahawk_channel_topology_run_manifest.json
  /kaggle/working/tomahawk_channel_topology_wall_vs_dp.png


,topology,cases,passing_cases,minimum_wall_temp_C,maximum_wall_temp_C,minimum_pressure_drop_kPa,maximum_pressure_drop_kPa,maximum_velocity_m_s,pass_fraction
0,circular_control,13122,1557,132.478,576.755,0.309,41043.496,127.573,0.119
1,microchannel_stress_test,13122,1956,130.736,523.642,0.926,123130.487,127.573,0.149
2,parallel_ribbed,13122,1880,131.067,533.734,0.494,65669.593,127.573,0.143
3,parallel_smooth,13122,1625,132.184,567.792,0.309,41043.496,127.573,0.124


## Recommended interpretation

Use the best passing `parallel_smooth` case as the baseline CAD geometry. Compare it
against `parallel_ribbed` to quantify the benefit of localized mixing. Treat
`microchannel_stress_test` as a warning/control branch, not as the default design.

Before hardware decisions, replace the reduced-order assumptions with:

- CAD-derived channel area, wetted perimeter, roughness, and manifold losses.
- OpenMC/MCNP volumetric heating tallies.
- 3-D conjugate heat-transfer CFD.
- RF surface-current and dielectric-loss maps.
- Irradiated CuCrZr/AlN/braze property data and pressure-boundary analysis.